# Milestone 4 lab, notebook 5 of 7: Layers and the incremental model

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| **5. Layers and the incremental model** (you are here) | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

You have seen three models. The project has six, and a seed. This notebook shows how they are arranged, and then looks closely at the one model that does **not** rebuild itself from nothing every time.

In [1]:
from lab_helpers import MILESTONE_DIR, dbt, lab_warehouse, load_only, show_doc, table

con = lab_warehouse(fresh=True)
load_only(con, "2026-06-02", "2026-06-03", "2026-06-04", "2026-06-05", "2026-06-06", "2026-06-07", "2026-06-08")
dbt("build", show=False)
print("\nSeven mornings are landed, and the models are built:", dbt.passed)

Landed  28 raw rows for the morning of 2026-06-02.
Landed  60 raw rows for the morning of 2026-06-03.
Landed  96 raw rows for the morning of 2026-06-04.
Landed 107 raw rows for the morning of 2026-06-05.
Landed 109 raw rows for the morning of 2026-06-06.
Landed 105 raw rows for the morning of 2026-06-07.
Landed 105 raw rows for the morning of 2026-06-08.



Seven mornings are landed, and the models are built: True


## Step 1: Three layers

The models are in three folders, and each folder is a **layer**: a set of tables at one level of cleaning.

In [2]:
show_doc("HLD.md", "3. Three layers")

**From `docs/HLD.md`**

> ## 3. Three layers
> 
> A layer is a set of tables at one level of cleaning. Each layer has one job.
> 
> | Layer | Its job | Here | Built as |
> |---|---|---|---|
> | **raw** | Keep what arrived, exactly | `orders_landing`, `order_deletes` | Tables. Python appends |
> | **staging** | Clean it: types, one spelling, one row per key | `stg_orders`, `stg_order_items`, `stg_order_deletes` | Views |
> | **core** | One row per real thing | `orders`, `order_items`, `cities` | Tables |
> | **marts** | What people read | `daily_orders` | Incremental table |
> 
> Other teams call the same idea bronze, silver and gold: raw is bronze, the cleaned tables are silver, and the business-ready tables are gold. The names are a habit, not a rule.
> 
> The rule that matters: **clean first, then shape, then add up.** Each layer reads only from the layer before it.

Follow one order down through the layers.

In [3]:
table(con, """
    SELECT '1 raw' AS layer, 'orders_landing' AS "table", count(*) AS rows_for_order_1130, 'every version, as JSON' AS holds
    FROM raw.orders_landing WHERE (payload->>'order_id')::bigint = 1130
    UNION ALL SELECT '2 staging', 'stg_orders', count(*), 'the newest version, typed and clean' FROM staging.stg_orders WHERE order_id = 1130
    UNION ALL SELECT '3 core', 'orders', count(*), 'the order, with its city name and delete flag' FROM core.orders WHERE order_id = 1130
    UNION ALL SELECT '3 core', 'order_items', count(*), 'its item lines' FROM core.order_items WHERE order_id = 1130
    UNION ALL SELECT '4 marts', 'daily_orders', count(*), 'the day and city it is counted in' FROM marts.daily_orders WHERE order_date = '2026-06-05' AND city = 'NGT'
    ORDER BY 1, 2
""")

,layer,table,rows_for_order_1130,holds
0,1 raw,orders_landing,3,"every version, as JSON"
1,2 staging,stg_orders,1,"the newest version, typed and clean"
2,3 core,order_items,4,its item lines
3,3 core,orders,1,"the order, with its city name and delete flag"
4,4 marts,daily_orders,1,the day and city it is counted in


Each layer reads only from the layer before it. That gives one rule for where a change goes:

| You want to change | The layer |
|---|---|
| How a raw value is read or spelled | staging |
| What an order **is**: which columns, which joins | core |
| What the manager sees | marts |

dbt knows the order to build in, from the `ref()` calls. Ask it.

In [4]:
dbt("ls", "--resource-type", "model", "seed", "--output", "path")

models/marts/daily_orders.sql
models/core/order_items.sql
models/core/orders.sql
models/staging/stg_order_deletes.sql
models/staging/stg_order_items.sql
models/staging/stg_orders.sql
seeds/cities.csv


## Step 2: The report does not rebuild everything

`orders` and `order_items` are built from scratch on every run. With a few hundred orders that takes no time.

The report is different. Most of it never changes: last week's days are final. Rebuilding them every morning is wasted work, and on a real warehouse it is wasted money. So `daily_orders` is an **incremental model**.

In [5]:
print((MILESTONE_DIR / "dbt_project" / "models" / "marts" / "daily_orders.sql").read_text())

-- Mart: the manager's report. One row per business day, per city.
--
-- An incremental model. The first run builds every day. Each later run rebuilds only
-- the newest days: it deletes those days and inserts them again. Older days are not touched.

{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'delete+insert',
        unique_key = 'order_date'
    )
}}

select
    order_date,
    city,
    city_name,
    count(*) filter (where status <> 'cancelled')::int                        as orders,
    count(*) filter (where status = 'cancelled')::int                         as cancelled_orders,
    coalesce(sum(net_cents) filter (where status <> 'cancelled'), 0)::bigint  as revenue_cents,
    coalesce(sum(refunded_cents) filter (where status <> 'cancelled'), 0)::bigint as refunded_cents
from {{ ref('orders') }}
where not is_deleted                              -- a deleted order is not counted

{% if is_incremental() %}
  -- Only the days from N days befor

Two parts are new.

**The `config` block** says how to build it: `incremental`, with the strategy `delete+insert`, and `order_date` as the key. In words: *for the days this run produces, delete those days from the table, then insert them.* That is milestone 3's "replace one day".

**The `{% if is_incremental() %}` block** says which days this run produces. The table is called `{{ this }}`. The block keeps only days from 7 days before the newest day already in the report.

- On the first run the table does not exist. `is_incremental()` is false, the block is left out, and **every** day is built.
- On every later run the block is in, and only the newest days are built.

## Step 3: See which days a run touches

To see it, mark every row of the report with an impossible number. Then run the model, and look for the marks that are left.

In [6]:
con.execute("UPDATE marts.daily_orders SET cancelled_orders = 777")

dbt("run", "--select", "daily_orders")

table(con, """
    SELECT order_date, CASE WHEN max(cancelled_orders) = 777 THEN 'not touched' ELSE 'rebuilt' END AS this_run
    FROM marts.daily_orders GROUP BY order_date ORDER BY order_date
""")

1 of 1 OK created sql incremental model marts.daily_orders ..................... [INSERT 0 21]
Completed successfully
Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


,order_date,this_run
0,2026-06-01,rebuilt
1,2026-06-02,rebuilt
2,2026-06-03,rebuilt
3,2026-06-04,rebuilt
4,2026-06-05,rebuilt
5,2026-06-06,rebuilt
6,2026-06-07,rebuilt


The report's newest day is Sunday 7 June. Seven days before that is 31 May, so this small report is rebuilt whole. Load more mornings, and the window starts to leave the oldest days alone.

In [7]:
dbt("run", "--select", "daily_orders", "--full-refresh", show=False)       # remove the marks
load_only(con, "2026-06-09", "2026-06-10", "2026-06-11", "2026-06-12", "2026-06-13", "2026-06-14", "2026-06-15")
dbt("build", show=False)

con.execute("UPDATE marts.daily_orders SET cancelled_orders = 777")
dbt("run", "--select", "daily_orders", show=False)

table(con, """
    SELECT order_date, CASE WHEN max(cancelled_orders) = 777 THEN 'not touched' ELSE 'rebuilt' END AS this_run
    FROM marts.daily_orders GROUP BY order_date ORDER BY order_date
""")

Landed 102 raw rows for the morning of 2026-06-09.
Landed  93 raw rows for the morning of 2026-06-10.
Landed 120 raw rows for the morning of 2026-06-11.
Landed 103 raw rows for the morning of 2026-06-12.
Landed  93 raw rows for the morning of 2026-06-13.
Landed  90 raw rows for the morning of 2026-06-14.
Landed  90 raw rows for the morning of 2026-06-15.


,order_date,this_run
0,2026-06-01,not touched
1,2026-06-02,not touched
2,2026-06-03,not touched
3,2026-06-04,not touched
4,2026-06-05,not touched
5,2026-06-06,not touched
6,2026-06-07,rebuilt
7,2026-06-08,rebuilt
8,2026-06-09,rebuilt
9,2026-06-10,rebuilt


The newest day is now 14 June. The run rebuilt 7 June to 14 June, and did not touch 1 June to 6 June.

## Step 4: Compare it with its opposite, a full refresh

The marks are still on the old days: the model left them alone, so it left the damage too. The opposite of an incremental run is a **full refresh**: build every day again.

In [8]:
dbt("run", "--select", "daily_orders", "--full-refresh")

table(con, "SELECT count(DISTINCT order_date) AS days, count(*) FILTER (WHERE cancelled_orders = 777) AS marks_left FROM marts.daily_orders")

1 of 1 OK created sql incremental model marts.daily_orders ..................... [SELECT 42]
Completed successfully
Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


,days,marks_left
0,14,0


| | Incremental run | Full refresh |
|---|---|---|
| Days built | The newest 8 | All 14 |
| Old days | Left alone | Rebuilt |
| Use it | Every morning | After a rule changed, or to repair old days |

The two must agree. A test in this milestone builds the report incrementally over 17 mornings, then does a full refresh, and checks that nothing changed.

## Step 5: Why 7 days?

The window must be wide enough to catch every change that can still move a day. Look at Arden's data: how long after an order was placed did it last change?

In [9]:
table(con, """
    SELECT max(updated_at::date - order_date) AS most_days_between_order_and_last_change,
           count(*) FILTER (WHERE updated_at::date - order_date > 1) AS orders_changed_after_more_than_a_day
    FROM core.orders
""")

,most_days_between_order_and_last_change,orders_changed_after_more_than_a_day
0,3,1


The latest change came 3 days after the order: the refund on order 1130. Seven days leaves room.

It is still a limit. A refund 10 days later would not reach the report through an incremental run. That is why there is a test that compares the report with its orders: it would fail, and a full refresh would repair it. You meet that test in the next notebook.

In [10]:
con.close()
show_doc("LLD.md", "5. Mart: the incremental model (`daily_orders.sql`)")

**From `docs/LLD.md`**

> ## 5. Mart: the incremental model (`daily_orders.sql`)
> 
> ```sql
> {{ config(materialized='incremental', incremental_strategy='delete+insert', unique_key='order_date') }}
> 
> select order_date, city, city_name, count(*) filter (...) as orders, ...
> from {{ ref('orders') }}
> where not is_deleted
> {% if is_incremental() %}
>   and order_date >= (select max(order_date) from {{ this }}) - {{ var('reprocess_days') }}
> {% endif %}
> group by order_date, city, city_name
> ```
> 
> | Run | `is_incremental()` | What dbt does |
> |---|---|---|
> | The first run, or `--full-refresh` | false | Builds every day |
> | Every later run | true | Works out the days from 7 days before the report's newest day. Deletes those days from the table. Inserts them again |
> 
> It is milestone 3's "replace one day", written as three settings. Measured on the run of 16 June: the report holds 14 days, and the run rebuilds the last 8 of them.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Staging, core, mart** | Three layers of models. Staging cleans the raw data. Core holds one row per real thing. A mart is what people read | `stg_orders`, then `orders`, then `daily_orders` |
| **Incremental model** | A model that, after its first run, rebuilds only part of its table | `daily_orders`: the newest days, deleted and inserted again |

Other teams call the same layers bronze, silver and gold. The layer idea is in the D8Loop chapters [The Data Pipeline Lifecycle](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/the-data-pipeline-lifecycle) and [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals). Incremental processing is in [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills).

## What you now know

- Staging cleans, core holds one row per real thing, and a mart is what people read.
- dbt works out the build order from `ref()`.
- An incremental model rebuilds only the newest days. A full refresh rebuilds all of them.
- The window is a decision. Choose it from how late a day can still change.

**Say it in an interview:** "My report is an incremental dbt model with the delete+insert strategy and the business day as the key. Each run rebuilds the last week and leaves older days alone. I chose seven days from the data: the latest change came three days after the order. And I have a test that the incremental result equals a full refresh."

Next: [notebook 6](06_tests_on_the_tables.ipynb) puts checks on the tables, and breaks the data on purpose.